In [5]:
import gmsh

gmsh.initialize()

gmsh.model.add("two_wires")

# --------------------------------------------------
# Parameters
# --------------------------------------------------

L = 2.0                 # air box

r = 0.05               # wire radius

distance = 0.70         # center-to-center spacing

cx = L / 2
cy = L / 2

x1 = cx - distance / 2
x2 = cx + distance / 2

lc_air = 0.05
lc_wire = 0.01

# --------------------------------------------------
# Air domain
# --------------------------------------------------

square = gmsh.model.occ.addRectangle(
    0,
    0,
    0,
    L,
    L
)

# --------------------------------------------------
# Wire 1
# --------------------------------------------------

wire1 = gmsh.model.occ.addDisk(
    x1,
    cy,
    0,
    r,
    r
)

# --------------------------------------------------
# Wire 2
# --------------------------------------------------

wire2 = gmsh.model.occ.addDisk(
    x2,
    cy,
    0,
    r,
    r
)

# --------------------------------------------------
# Remove both wires from air
# --------------------------------------------------

air, _ = gmsh.model.occ.cut(
    [(2, square)],
    [(2, wire1), (2, wire2)],
    removeObject=True,
    removeTool=False
)

gmsh.model.occ.synchronize()

# --------------------------------------------------
# Physical groups
# --------------------------------------------------

gmsh.model.addPhysicalGroup(
    2,
    [air[0][1]],
    name="air"
)

gmsh.model.addPhysicalGroup(
    2,
    [wire1],
    name="wire1"
)

gmsh.model.addPhysicalGroup(
    2,
    [wire2],
    name="wire2"
)

# Outer boundary (Dirichlet BC)
boundary = gmsh.model.getBoundary(
    [(2, air[0][1])],
    oriented=False,
    recursive=False
)

gmsh.model.addPhysicalGroup(
    1,
    [b[1] for b in boundary],
    name="boundary"
)

# --------------------------------------------------
# Mesh sizes
# --------------------------------------------------

gmsh.model.mesh.setSize(
    gmsh.model.getEntities(0),
    lc_air
)

# Refine around wire 1
wire1_boundary = gmsh.model.getBoundary(
    [(2, wire1)],
    recursive=True
)

gmsh.model.mesh.setSize(
    wire1_boundary,
    lc_wire
)

# Refine around wire 2
wire2_boundary = gmsh.model.getBoundary(
    [(2, wire2)],
    recursive=True
)

gmsh.model.mesh.setSize(
    wire2_boundary,
    lc_wire
)

# --------------------------------------------------
# Generate mesh
# --------------------------------------------------

gmsh.model.mesh.generate(2)

gmsh.write("two_wires.msh")

gmsh.fltk.run()

gmsh.finalize()